# SQLmind Lab 4: The PostgreSQL Production Agent

**Goal**: Connect to a real PostgreSQL database where the AI has **zero hardcoded knowledge** of the schema.
The agent will autonomously explore tables and generate correct SQL using:
- `sql_db_list_tables` → Discover what tables exist
- `sql_db_schema` → Read the column definitions + sample rows
- `sql_db_query_checker` → Validate the SQL before running it
- `sql_db_query` → Execute the final, validated query


In [ ]:
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_ollama import ChatOllama

# ==========================================
# CELL 1: CONNECT TO POSTGRESQL
# ==========================================
# Update with your actual PostgreSQL credentials:
# Format: postgresql+psycopg2://<user>:<password>@<host>:<port>/<database>

DB_URI = 

print("Connecting to PostgreSQL...")
try:
    db = SQLDatabase.from_uri(DB_URI)
    print(f"✅ Connected! Dialect: {db.dialect}")
    print(f"📋 Tables found: {db.get_usable_table_names()}")
except Exception as e:
    print(f"❌ Connection Failed: {e}")
    print("👉 Make sure PostgreSQL is running and your credentials in DB_URI are correct.")

# Setup the Brain (same model as before)
MODEL_NAME = "qwen3.5:9b"
llm = ChatOllama(model=MODEL_NAME, temperature=0)
print(f"\n🧠 Brain Loaded: {MODEL_NAME}")

Connecting to PostgreSQL...
✅ Connected! Dialect: postgresql
📋 Tables found: ['tbl_property_details', 'tbl_property_tax_demand', 'tbl_tax_collections', 'tbl_total_tax_due', 'tbl_usage_sub_type_master', 'tbl_usage_type_master']

🧠 Brain Loaded: qwen3.5:9b


In [10]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit

# ==========================================
# CELL 2: BUILD THE OFFICIAL TOOLKIT
# ==========================================

# This single line packages ALL 4 discovery + execution tools automatically
toolkit = SQLDatabaseToolkit(db=db, llm=llm)
tools = toolkit.get_tools()

print("🛠️  SQL Toolkit Loaded! Available Tools:")
for t in tools:
    print(f"   - {t.name}: {t.description[:80]}...")

🛠️  SQL Toolkit Loaded! Available Tools:
   - sql_db_query: Input to this tool is a detailed and correct SQL query, output is a result from ...
   - sql_db_schema: Input to this tool is a comma-separated list of tables, output is the schema and...
   - sql_db_list_tables: Input is an empty string, output is a comma-separated list of tables in the data...
   - sql_db_query_checker: Use this tool to double check if your query is correct before executing it. Alwa...


In [11]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from pydantic import BaseModel, Field

# ==========================================
# CELL 3: THE LANGGRAPH FACTORY (UPGRADED)
# ==========================================

# --- 1. AgentState: The Clipboard ---
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    needs_sql: bool

# --- 2. The Pydantic Router Contract (same as before) ---
class QueryAnalyzer(BaseModel):
    is_sql_required: bool = Field(
        description="True ONLY if the question asks about data in a database. Otherwise, False."
    )
    reasoning: str = Field(
        description="One-sentence explanation of why."
    )

system_prompt = SystemMessage(
    content="""You are a linguistic analysis algorithm, not an AI assistant.
    Your ONLY function is to determine if a user's question requires querying a database.
    If the question asks about data, records, statistics, or information that would live in a database, set is_sql_required to True.
    If it is a general question or casual conversation, set is_sql_required to False."""
)
structured_analyzer = llm.with_structured_output(QueryAnalyzer)

# --- 3. Node 1: Traffic Cop (same smart router) ---
def traffic_cop_node(state: AgentState):
    user_text = state["messages"][-1].content
    print(f"\n[🚦 Traffic Cop] Analyzing: '{user_text}'")
    try:
        result = structured_analyzer.invoke([system_prompt, HumanMessage(content=user_text)])
        print(f"[🚦 Traffic Cop] Decision -> Needs SQL: {result.is_sql_required}")
        return {"needs_sql": result.is_sql_required}
    except Exception:
        print("[🚦 Traffic Cop] ⚠️ Routing Error. Defaulting to standard chat.")
        return {"needs_sql": False}

# --- 4. Node 2: Standard Chatbot (unchanged) ---
def standard_chat_node(state: AgentState):
    print("[💬 Chatbot] Generating normal friendly response...")
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

# --- 5. Node 3: The SQL Specialist (UPGRADED with toolkit) ---
# Bind the official toolkit to the LLM
sql_agent_llm = llm.bind_tools(tools)

SQL_SYSTEM_PROMPT = """You are an expert SQL agent connected to a PostgreSQL database.
You have NO prior knowledge of the database schema. You MUST follow these steps in order:
1. FIRST call 'sql_db_list_tables' to discover what tables exist.
2. THEN call 'sql_db_schema' on the relevant table(s) to read the column names and sample data.
3. THEN call 'sql_db_query_checker' to validate your SQL before running it.
4. FINALLY call 'sql_db_query' to execute the validated query and get the answer.
Never skip any of these steps."""

def sql_specialist_node(state: AgentState):
    print("[🛠️  SQL Specialist] Taking over. Starting schema discovery...")
    messages_to_send = [SystemMessage(content=SQL_SYSTEM_PROMPT)] + state["messages"]
    response = sql_agent_llm.invoke(messages_to_send)
    return {"messages": [response]}

# Tool executor node — runs whatever tool the AI called
tool_node = ToolNode(tools)

# --- 6. Routing Functions ---
def route_decision(state: AgentState):
    """Route after the traffic cop based on needs_sql flag."""
    return "sql_specialist" if state["needs_sql"] else "standard_chat"

def should_continue_sql(state: AgentState):
    """Keep looping the SQL agent until it stops calling tools."""
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        print(f"[🔄 Loop] AI called: {[tc['name'] for tc in last_message.tool_calls]}")
        return "tools"
    print("[✅ SQL Specialist] Done! Returning final answer.")
    return END

# --- 7. Build the Graph ---
builder = StateGraph(AgentState)

builder.add_node("traffic_cop", traffic_cop_node)
builder.add_node("standard_chat", standard_chat_node)
builder.add_node("sql_specialist", sql_specialist_node)
builder.add_node("tools", tool_node)  # NEW: handles all 4 toolkit tool calls

builder.add_edge(START, "traffic_cop")
builder.add_conditional_edges("traffic_cop", route_decision)
builder.add_edge("standard_chat", END)
# The SQL specialist loops back through tools until it has a final answer
builder.add_conditional_edges("sql_specialist", should_continue_sql)
builder.add_edge("tools", "sql_specialist")  # After tool runs, go back to sql_specialist

sqlmind_postgres_app = builder.compile()
print("✅ Graph compiled! The production agent is ready.")

✅ Graph compiled! The production agent is ready.


In [18]:
# ==========================================
# CELL 4: LIVE TEST — WATCH THE AGENT EXPLORE AND THINK
# ==========================================
# The agent starts with ZERO knowledge of your PostgreSQL schema.

test_questions = [
#    "What tables are available in the database?",
    "tax collection against tax demand financial year wise"
]

for q in test_questions:
    print("\n" + "="*70)
    print(f"USER: {q}")
    initial_state = {"messages": [HumanMessage(content=q)]}
    
    # We use .stream() instead of .invoke() to watch the agent step-by-step
    final_state = None
    for event in sqlmind_postgres_app.stream(initial_state):
        for node_name, node_state in event.items():
            if "messages" in node_state:
                latest_msg = node_state["messages"][-1]
                
                # 1. Did the AI decide to use a tool? (e.g. write a SQL query)
                if hasattr(latest_msg, "tool_calls") and latest_msg.tool_calls:
                    for tc in latest_msg.tool_calls:
                        print(f"\n[AI Thinking] Decided to use tool: '{tc['name']}'")
                        print(f"   └── Input Arguments: {tc['args']}")
                
                # 2. Did a Tool just finish running? (e.g. DB returned results)
                elif latest_msg.type == "tool":
                    print(f"\n[Tool Execution] '{latest_msg.name}' returned results:")
                    res_preview = str(latest_msg.content)[:400] + ("..." if len(str(latest_msg.content)) > 400 else "")
                    print(f"   └── {res_preview}")
            
            # Keep track of the final state to print the final answer
            final_state = node_state
            
    print(f"\n[Final Answer]: {final_state['messages'][-1].content}")



USER: tax collection against tax demand financial year wise

[🚦 Traffic Cop] Analyzing: 'tax collection against tax demand financial year wise'
[🚦 Traffic Cop] Decision -> Needs SQL: True
[🛠️  SQL Specialist] Taking over. Starting schema discovery...
[🔄 Loop] AI called: ['sql_db_list_tables']

[AI Thinking] Decided to use tool: 'sql_db_list_tables'
   └── Input Arguments: {'tool_input': ''}

[Tool Execution] 'sql_db_list_tables' returned results:
   └── tbl_property_details, tbl_property_tax_demand, tbl_tax_collections, tbl_total_tax_due, tbl_usage_sub_type_master, tbl_usage_type_master
[🛠️  SQL Specialist] Taking over. Starting schema discovery...
[🔄 Loop] AI called: ['sql_db_schema']

[AI Thinking] Decided to use tool: 'sql_db_schema'
   └── Input Arguments: {'table_names': 'tbl_property_tax_demand, tbl_tax_collections'}

[Tool Execution] 'sql_db_schema' returned results:
   └── 
CREATE TABLE tbl_property_tax_demand (
	demand_id SERIAL NOT NULL, 
	new_property_no VARCHAR(50) NOT NUL